# DDRNet-23-slim 訓練手冊 — Google Colab (T4 GPU)

**預計時間**：Step 4b 預計算 ~20 min + 訓練 ~6–10 小時（150 epochs, batch=32, T4）  
**本機同步**：本機 DeepLabV3+ 訓練中，兩者可同步平行執行  

---

## 使用前準備（只需做一次）

### 本機打包並上傳到 Google Drive

在 `H:\ChihleeMaster\dev\ABECIS_MODEL_SWAP` 開 Git Bash 執行：

```bash
# 1. 打包原始資料集（1.1GB → 壓縮後約 700MB）
tar -czf dataset.tar.gz concreteCrackSegmentationDataset/

# 2. 打包 splits（只有 3 個 txt，幾 KB）
tar -czf data_splits.tar.gz data/splits/
```

將 `dataset.tar.gz` 和 `data_splits.tar.gz` 上傳到 Google Drive：`My Drive/ABECIS/`

> **效能說明**：Step 4b 會在本地 SSD（`/content/patches/`，~35 GB）預計算所有 patches，  
> 之後訓練讀取 .npy 而非即時從圖片切割，速度提升約 10–20x（從 5s/it → ~0.3s/it）。  
> Colab 重啟需重新執行 Step 4b（~20 min），但訓練速度大幅改善。

---

## Step 0 — 確認 GPU 類型

In [ ]:
import subprocess, torch
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print(result.stdout)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.version.cuda}")
print(f"GPU:     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND'}")
# 目標：T4 (15GB) 或 L4 (22GB)
# 若出現 K80 → Runtime > Change runtime type > T4

## Step 1 — 掛載 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/ABECIS'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f"Drive root: {DRIVE_ROOT}")
print(os.listdir(DRIVE_ROOT))

## Step 2 — Clone 專案 repo 與 zh320 模型庫

In [ ]:
import os
os.chdir('/content')

# 指定 traing/deeplabv3 分支（含 DDRNet importlib fix，main 尚未合併）
!git clone -b traing/deeplabv3 https://github.com/penguin789456/ABECIS_MODEL_SWAP.git
!git clone https://github.com/zh320/realtime-semantic-segmentation-pytorch.git \
    /content/ABECIS_MODEL_SWAP/realtime-semantic-segmentation-pytorch

os.chdir('/content/ABECIS_MODEL_SWAP')
print("Working dir:", os.getcwd())
!git branch  # 應顯示 * traing/deeplabv3

## Step 3 — 安裝相依套件

In [ ]:
!pip install -q albumentations==1.4.21 scikit-image loguru tqdm pyyaml

import albumentations
print(f"albumentations: {albumentations.__version__}  (需 1.4.21)")
import torch
print(f"torch: {torch.__version__}")

## Step 4 — 解壓資料集與 Splits

In [ ]:
import os
os.chdir('/content/ABECIS_MODEL_SWAP')
DRIVE_ROOT = '/content/drive/MyDrive/ABECIS'

if not os.path.exists('concreteCrackSegmentationDataset/rgb'):
    print("解壓資料集（約 2–3 分鐘）...")
    !tar -xzf {DRIVE_ROOT}/dataset.tar.gz
else:
    print("資料集已存在，跳過")

if not os.path.exists('data/splits/train.txt'):
    print("解壓 splits...")
    !tar -xzf {DRIVE_ROOT}/data_splits.tar.gz
else:
    print("Splits 已存在，跳過")

for path in ['data/splits/train.txt', 'data/splits/val.txt', 'data/splits/test.txt',
             'concreteCrackSegmentationDataset/rgb']:
    print(f"{'✅' if os.path.exists(path) else '❌'} {path}")

## Step 4b — 預計算 Patches 到本地 SSD

> **必須執行，大幅提速**：將 458 張原始圖片切割為 512×512 patches，存至 `/content/patches/`（本地 SSD）。  
> 約需 **15–20 分鐘**，但訓練速度從 ~5s/it 降至 ~0.3s/it（快約 **15x**）。  
> Colab 重啟後需重新執行此步（patches 存在本地，不佔 Drive 空間）。
>
> 磁碟需求：train+val+test ≈ 35–40 GB（Colab 本地磁碟約 107 GB，空間足夠）。

In [ ]:
import os, time
os.chdir('/content/ABECIS_MODEL_SWAP')

PATCHES_DIR = '/content/patches'

# 確認 manifest：若已預計算則跳過
train_manifest = f'{PATCHES_DIR}/train/manifest.json'
if os.path.exists(train_manifest):
    print(f"✅ Patches 已存在於 {PATCHES_DIR}，跳過")
    !du -sh {PATCHES_DIR}
else:
    print(f"開始預計算 patches → {PATCHES_DIR}")
    print("預計時間：15–20 分鐘...")
    t0 = time.time()

    # 先將 precomputed_dir 設為 /content/patches，再執行 precompute
    import yaml
    with open('configs/final/ddrnet.yaml', encoding='utf-8') as f:
        _cfg = yaml.safe_load(f)
    _cfg['dataset']['precomputed_dir'] = PATCHES_DIR
    with open('/tmp/ddrnet_precompute.yaml', 'w', encoding='utf-8') as f:
        yaml.dump(_cfg, f, allow_unicode=True)

    !python scripts/precompute_patches.py --config /tmp/ddrnet_precompute.yaml

    elapsed = time.time() - t0
    print(f"\n✅ 預計算完成，耗時 {elapsed/60:.1f} 分鐘")
    !du -sh {PATCHES_DIR}
    !ls {PATCHES_DIR}/train/ | head -5

## Step 5 — 寫入 Colab 用 Config

> `precomputed_dir` 設為 `/content/patches`，訓練使用預計算的 .npy 檔（快 15x）。

In [ ]:
import yaml, os
DRIVE_ROOT = '/content/drive/MyDrive/ABECIS'
PATCHES_DIR = '/content/patches'

with open('configs/final/ddrnet.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

# ── 核心：使用預計算 patches（本地 SSD，快 15x）──
cfg['dataset']['precomputed_dir'] = PATCHES_DIR

# Colab 環境調整
cfg['dataset']['num_workers'] = 4
cfg['dataset']['persistent_workers'] = True
cfg['dataset']['prefetch_factor'] = 2
cfg['dataset']['pin_memory'] = True

# checkpoint + CSV log → 直接存 Drive（防斷線遺失）
cfg['checkpoint']['save_dir'] = f'{DRIVE_ROOT}/checkpoints/ddrnet'
cfg['training']['output_dir'] = f'{DRIVE_ROOT}/outputs'

os.makedirs(f'{DRIVE_ROOT}/checkpoints/ddrnet', exist_ok=True)

with open('configs/colab_ddrnet.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(cfg, f, allow_unicode=True)

print("✅ configs/colab_ddrnet.yaml 寫入完成")
print(f"   precomputed_dir : {cfg['dataset']['precomputed_dir']}")
print(f"   num_workers     : {cfg['dataset']['num_workers']}")
print(f"   checkpoint      → {cfg['checkpoint']['save_dir']}")
print(f"   batch_size      : {cfg['training']['batch_size']}")
print(f"   epochs          : {cfg['training']['epochs']}")

## Step 6 — 開始訓練

`best.pth` 和 `train_log.csv` 每個 epoch 後自動寫入 Drive，斷線不遺失。

In [ ]:
import os
os.chdir('/content/ABECIS_MODEL_SWAP')

!python training/train_crackseg.py --config configs/colab_ddrnet.yaml 2>&1 | tee /content/ddrnet_train.log

## Step 6b — Resume 訓練（Colab 斷線後使用）

> 斷線後重新執行 **Step 1–5**（解壓 ~3 min + 預計算 ~20 min），再執行此 Cell。  
> Checkpoint 在 Drive 上，從最後儲存的 epoch 繼續。

In [ ]:
import os, glob, torch
DRIVE_ROOT = '/content/drive/MyDrive/ABECIS'
os.chdir('/content/ABECIS_MODEL_SWAP')

best_pth = f'{DRIVE_ROOT}/checkpoints/ddrnet/best.pth'
if os.path.exists(best_pth):
    info = torch.load(best_pth, map_location='cpu')
    print(f"Resume from: {best_pth}")
    print(f"  已完成 epoch: {info['epoch']}  best IoU: {info['best_iou']:.4f}")
    !python training/train_crackseg.py \
        --config configs/colab_ddrnet.yaml \
        --resume "{best_pth}" \
        2>&1 | tee /content/ddrnet_resume.log
else:
    print("❌ Drive 上找不到 best.pth")

## Step 7 — 訓練結果確認

In [ ]:
import pandas as pd, glob, os
DRIVE_ROOT = '/content/drive/MyDrive/ABECIS'

logs = glob.glob(f'{DRIVE_ROOT}/checkpoints/ddrnet/**/train_log.csv', recursive=True)
if logs:
    df = pd.read_csv(logs[0])
    best = df.loc[df['iou'].idxmax()]
    print(f"總 epoch 數 : {len(df)}")
    print(f"最佳 val IoU: {best['iou']:.4f} @ epoch {int(best['epoch'])}")
    print(f"對應 Dice   : {best['dice']:.4f}")
    print(f"對應 Recall : {best['recall']:.4f}")
    print("\n最後 5 epochs:")
    print(df[['epoch','train_loss','val_loss','iou','dice','recall','lr']].tail().to_string(index=False))
else:
    print("❌ train_log.csv 不存在")

best_pth = f'{DRIVE_ROOT}/checkpoints/ddrnet/best.pth'
if os.path.exists(best_pth):
    print(f"\n✅ best.pth ({os.path.getsize(best_pth)/1e6:.1f} MB)")
else:
    print("\n❌ best.pth 不存在")

## Step 7b — 繪製訓練曲線

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, glob
DRIVE_ROOT = '/content/drive/MyDrive/ABECIS'

logs = glob.glob(f'{DRIVE_ROOT}/checkpoints/ddrnet/**/train_log.csv', recursive=True)
if not logs:
    print("❌ train_log.csv 不存在")
else:
    df = pd.read_csv(logs[0])
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(df['epoch'], df['train_loss'], label='Train')
    axes[0].plot(df['epoch'], df['val_loss'],   label='Val')
    axes[0].set_title('Loss'); axes[0].legend()
    axes[1].plot(df['epoch'], df['iou'],  label='IoU')
    axes[1].plot(df['epoch'], df['dice'], label='Dice')
    axes[1].set_title('Metrics'); axes[1].legend()
    axes[2].plot(df['epoch'], df['lr'])
    axes[2].set_title('Learning Rate'); axes[2].set_yscale('log')
    plt.tight_layout()
    out_png = f'{DRIVE_ROOT}/ddrnet_training_curve.png'
    plt.savefig(out_png, dpi=150)
    plt.show()
    print(f"✅ 圖表儲存到 Drive")

## Step 8 — Threshold Sweep（訓練完成後）

In [ ]:
import os
os.chdir('/content/ABECIS_MODEL_SWAP')
DRIVE_ROOT = '/content/drive/MyDrive/ABECIS'

!python scripts/threshold_sweep.py \
    --config configs/final/ddrnet.yaml \
    --checkpoint "{DRIVE_ROOT}/checkpoints/ddrnet/best.pth" \
    --min_t 0.45 --max_t 0.75 --step 0.025

print("\n▶ 將最佳 threshold 填入 configs/final/ddrnet.yaml → evaluation.threshold")

## Step 9 — 下載 best.pth 到本機

In [ ]:
from google.colab import files
import os
DRIVE_ROOT = '/content/drive/MyDrive/ABECIS'
best_pth = f'{DRIVE_ROOT}/checkpoints/ddrnet/best.pth'

if os.path.exists(best_pth):
    print(f"下載 best.pth ({os.path.getsize(best_pth)/1e6:.1f} MB)...")
    files.download(best_pth)
else:
    print("❌ best.pth 不存在")

---

## 完成後的本機操作

**1. 放置 checkpoint**
```
outputs/checkpoints/ddrnet/best.pth
```

**2. 填入最佳 threshold**（Step 8 sweep 結果）
```yaml
# configs/final/ddrnet.yaml
evaluation:
  threshold: 0.XX
```

**3. 執行統一評估**
```bash
scripts\run_eval_all.bat
```